# Assignment 1: A small language pipeline

You will turn a book PDF into a corpus, tokenise it, learn byte-pair encoding (BPE) and train n-gram language models. BPE here is the classic character-level algorithm, not byte-level BPE.

Write your code in `src/` and in the notebook cells marked `TODO`. Replace every `None` with a variable, a short expression or a short answer in quotes. Each `TODO` cell must print all of its results correctly to get the full mark. Submit the notebook with all outputs.

Use only the Python standard library, NumPy, pandas, Matplotlib and `pypdf`. Do not use NLP packages such as NLTK or spaCy, or any tokeniser, BPE or language-model library.

| Task | Marks |
|---|---:|
| 1. Corpus | 10 |
| 2. Tokenisation | 14 |
| 3. Data split | 6 |
| 4. Zipf plot and type-token ratio | 6 |
| 5. Learn BPE | 14 |
| 6. Apply BPE | 8 |
| 7. n-gram models | 18 |
| 8. Generate and evaluate | 12 |
| 9. Short questions | 12 |
| **Total** | **100** |

## Setup
Run this cell at the start of every session. In Colab it mounts Google Drive, moves to your copy of `assignment1` and installs the requirements. Change the path if you cloned elsewhere. Locally it only reloads `src/` automatically after you edit it. See the README for first-time setup.

In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/cng463-assignments/assignment1
%pip install -q -r requirements.txt
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import Counter
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
from src.text import (extract_pages, clean_pages, normalise_text, split_into_sentences, tokenise,
                        save_sentences, load_sentences, deduplicate, split_blocks, corpus_stats)
from src.bpe import learn_bpe, encode_word
from src.lm import NGramLM, BOS, EOS, UNK

## 1. Corpus (10 marks)
Choose a book in PDF form with a text layer. Scanned pages will not work. Public-domain or openly licensed books are the safest choice, for example from Project Gutenberg, Wikisource or the Internet Archive.

Choose a page range with 5,000–50,000 words and at least 200 sentences. Leave out front matter such as the contents and preface. Put the PDF in `data/` (2).

Implement `clean_pages`, `normalise_text` (4) and `split_into_sentences` in `src/text.py` (4). Cleaning should remove page numbers, running headers and headings, rejoin words split by a hyphen at a line end, and replace ligatures (single characters that combine multiple letters, such as ﬁ) with separate letters (e.g. ﬁ → fi).

In [ ]:
# TODO [Task 1]: Set your PDF and page range, then describe the book.
PDF_PATH, FIRST_PAGE, LAST_PAGE = "data/your_book.pdf", None, None
raw_pages = extract_pages(PDF_PATH, FIRST_PAGE, LAST_PAGE)
print("Title and author:", None)
print("Source:", None)
print("Licence:", None)
print("Language:", None)

In [ ]:
# TODO [Task 1]: Clean the pages and compare one raw page with its cleaned text.
pages = clean_pages(raw_pages)
PAGE = None  # index in raw_pages of a page with a running header and a hyphenated line break
print("RAW\n" + raw_pages[PAGE])
print("\nCLEANED\n" + normalise_text([pages[PAGE]]))
print("\nRunning header removed:", None)
print("Hyphenated word rejoined:", None)

In [ ]:
# TODO [Task 1]: Segment and save the sentences, then inspect them.
sentence_texts = split_into_sentences(normalise_text(pages))
save_sentences(sentence_texts, "data/corpus.txt")
for s in random.Random(0).sample(sentence_texts, 5):
    print("-", s)
print("\nSentences:", None)
print("Words:", None)
print("\nWrongly segmented sentence:", None)
print("Cause:", None)

## 2. Tokenisation (14 marks)
Implement `tokenise` in `src/text.py` with regular expressions (6). Show at least four edge cases, preferably from your book (4). Explain how you handle case, punctuation, apostrophes and numbers, and give one limitation for your corpus language (4).

In [ ]:
# TODO [Task 2]: Tokenise four sentences from your book and report one bad case.
examples = [None, None, None, None]
for text in examples:
    print(repr(text), "->", tokenise(text))
print()
for text in ["Don't", "3.50", "state-of-the-art", "İstanbul"]:
    print(repr(text), "->", tokenise(text))
print("\nString from the book handled badly:", None)
print("Why:", None)

## 3. Data split (6 marks)
Nearby sentences in a book share names and topics, so a random sentence split leaks information into the test set. Implement `deduplicate` (2) and `split_blocks` (2) in `src/text.py` (2).

In [ ]:
# TODO [Task 3]: Remove duplicates, split by blocks and check for overlap.
sentences = load_sentences("data/corpus.txt")
unique = deduplicate(sentences)
train, dev, test = split_blocks(unique, block_size=10, seed=42)
print("Duplicates removed:", None)
print("Train/dev/test sentences:", None, None, None)
print("Sentences shared by any two sets:", None)
print("Fitted on training data only:", None)
print("Set used to choose alpha:", None)

## 4. Zipf plot and type-token ratio (6 marks)
Implement `corpus_stats` and report N, V and TTR = V/N for the training set. Count every token, including punctuation (2). Plot frequency against rank with log axes and axis labels (2). Report the number of types that occur once and the TTR of the first 1,000 training tokens (2).

In [ ]:
# TODO [Task 4]: Compute training statistics and plot rank against frequency.
freq, N, V, ttr = corpus_stats(train)
print("N:", None, " V:", None, " TTR:", None)
# Plot frequency against rank here, with log axes and axis labels, then call plt.show().

print("Types that occur once:", None)
print("TTR of the first 1,000 training tokens:", None)

## 5. Learn BPE (14 marks)
Implement `merge_symbols` and `learn_bpe` in `src/bpe.py`. Start each training word type as its characters plus `</w>`.
- Count adjacent pairs inside each word, weighted by the word's training frequency (5).
- Pick the most frequent pair. Break ties by the smaller `(left, right)` tuple in Python's ordering, so `'</w>'` comes before letters. Merge left to right without overlaps (5).
- Learn 200 merges, show the first 10 with their counts and add one check that tests ties or overlaps (4).

In [ ]:
# TODO [Task 5]: Learn 200 merges, print the first 10 and test your code.
merges = learn_bpe(freq, num_merges=200)
print("First 10 merges:")
for i, (pair, count) in enumerate(merges[:10], 1):
    print(i, pair, count)
# Add an assert that tests ties or overlapping pairs on a small Counter you design.

print("Check passed:", None)

## 6. Apply BPE (8 marks)
Implement `encode_word` (3). Encode five training words and five test words, at least two of them unseen in training. Report the mean number of pieces per test token, counting `</w>` (3). Check that the pieces rebuild every test token, and give the number of pieces a character never seen in training produces (2).

In [ ]:
# TODO [Task 6]: Encode words, measure pieces per token and check reconstruction.
test_tokens = [t for s in test for t in s]
train_words = [None, None, None, None, None]
test_words = [None, None, None, None, None]
print("Unseen in training:", [w for w in test_words if w not in freq])
for word in train_words + test_words:
    print(word, "->", encode_word(word, merges))
print("Mean pieces per test token:", None)
print("All test tokens rebuilt:", None)
print("Pieces produced by an unseen character:", None)

## 7. n-gram models (18 marks)
Implement `NGramLM` in `src/lm.py` for n = 1, 2 and 3.
- Build the vocabulary from training data only. Keep tokens seen at least twice and map the rest to `<unk>`. Add `<unk>` and `</s>`. `<s>` is used only as context (4).
- Pad each sentence with n−1 `<s>` and end it with one `</s>`. Count each context and its next token, never across sentences (6).
- Use P(w | h) = (C(h, w) + α) / (C(h) + α|V|), with the same V for all orders (5).
- Check that the probabilities sum to one for a seen and an unseen context (3).

In [ ]:
# TODO [Task 7]: Fit the three models and check that each distribution sums to one.
models = {n: NGramLM(n, alpha=1.0).fit(train) for n in (1, 2, 3)}
for n, model in models.items():
    seen, unseen = None, None  # an (n-1)-token context from training and one that never occurs
    print(f"n = {n}: |V| =", None)
    print("  Sum over a seen context:", None)
    print("  Sum over an unseen context:", None)

## 8. Generate and evaluate (12 marks)
Implement `perplexity` and `generate` (6). Perplexity is exp(−mean log probability) over all predicted tokens, including one `</s>` per sentence but not `<s>`. Generation samples from the full smoothed distribution and stops at `</s>` or after 40 tokens.

For each order, choose α from {0.001, 0.01, 0.1, 1} by development perplexity. Report the test perplexity, the test unknown rate, the number of test sentences and tokens, the order with the lowest test perplexity, and two samples with seeds 7 and 21 (6).

In [ ]:
# TODO [Task 8]: Choose alpha on dev, evaluate once on test and generate.
rows = []
for n in (1, 2, 3):
    model = None  # the model with the lowest dev perplexity over alpha in (0.001, 0.01, 0.1, 1)
    models[n] = model
    rows.append((n, None, None, None, None))
    for seed in (7, 21):
        print(f"n = {n}, seed {seed}:", None)
results = pd.DataFrame(rows, columns=["n", "alpha", "dev perplexity", "test perplexity", "test unknown rate"])
print(results.round(3).to_string(index=False))
print("Test sentences:", None, " Test tokens:", None)
print("Order with the lowest test perplexity:", None)

## 9. Short questions (12 marks)
Answer each question with a number or a short sequence of symbols. Each answer earns one mark.
1. Apply the single merge (a, a) to the word `aaa`. Which symbols do you get?
2. How many merges does `learn_bpe` learn from the word counts {"ab": 1} before no pairs remain?
3. Over how many prediction events does your trigram test perplexity average?
4. As α grows very large, what value does every model's perplexity approach? Give the number for your models.
5. How many prediction events does one sentence of 12 word tokens add to the perplexity, for any n?
6. What is the probability that your selected unigram model generates an empty sentence? Round to three decimals.

In [ ]:
# TODO [Task 9]: Replace each None with your answer.
ANSWERS = {1: None, 2: None, 3: None, 4: None, 5: None, 6: None}
for q, answer in ANSWERS.items():
    print(f"Q{q}:", answer)